# neuroSAT — GPU training on Colab

Trains the two PyTorch models on a Colab **GPU** runtime:
* **Part A — Graph-Q-SAT / GAT-Q-SAT** (DQN + GNN, `GQSAT/dqn.py`).
* **Part B — AlphaZeroSAT** (Alpha(Go)Zero + MCTS, `AlphaZeroSAT/train_torch.py`).

Set the runtime to **GPU** first: *Runtime → Change runtime type → GPU*.
Checkpoints are written under the cloned repo and can be copied to Drive (last cell).

## 0. GPU check + clone the repo (everything is on GitHub)

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime → GPU'
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

import os
if not os.path.isdir('neuroSAT'):
    !git clone --recurse-submodules https://github.com/dmeoli/NeuroSAT neuroSAT
%cd neuroSAT
!git -C GQSAT rev-parse --abbrev-ref HEAD; git -C AlphaZeroSAT rev-parse --abbrev-ref HEAD

## 1. Install the modern stack
Colab already ships a CUDA `torch`; we only add the rest (no torch-scatter/sparse).

In [ ]:
!pip -q install torch-geometric gymnasium PyYAML tensorboardX scipy
!sudo apt-get -qq install -y swig zlib1g-dev >/dev/null
import sys
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
!sudo apt-get -qq install -y {PYV}-dev >/dev/null
print('build prereqs ready for', PYV)

## Part A — Graph-Q-SAT / GAT-Q-SAT (GPU)
### A.1 Build the MiniSat gym extension

In [ ]:
import sys, os
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
NPINC = __import__('numpy').get_include()
%cd /content/neuroSAT/GQSAT/minisat
!make clean >/dev/null 2>&1; true
!make python-wrap PYTHON={PYV} NUMPY_INC="{NPINC}" 2>&1 | tail -6
print('built:', os.path.exists('minisat/gym/_GymSolver.so'))
%cd /content/neuroSAT/GQSAT
import minisat  # registers sat-v0
from minisat.minisat.gym.GymSolver import sat  # noqa: F401
print('GQSAT env OK')

### A.2 Train GAT-Q-SAT on graph colouring (GPU)
Drop `--use_attention --heads 3` to train plain Graph-Q-SAT instead. Edit the
data paths / `--batch-updates` as needed. No `--no_cuda` ⇒ it uses the GPU.

In [ ]:
%cd /content/neuroSAT/GQSAT
TRAIN = '../data/graph-coloring/train/flat50-115'
VAL   = '../data/graph-coloring/val/flat50-115'
!python3 dqn.py \
  --logdir runs_gpu --env-name sat-v0 \
  --train-problems-paths {TRAIN} --eval-problems-paths {VAL} \
  --use_attention --heads 3 \
  --lr 0.00002 --bsize 64 --buffer-size 20000 \
  --eps-init 1.0 --eps-final 0.01 --eps-decay-steps 30000 --gamma 0.99 \
  --batch-updates 50000 --history-len 1 --init-exploration-steps 5000 \
  --step-freq 4 --target-update-freq 10 --loss mse --opt adam \
  --save-freq 500 --grad_clip 1 --grad_clip_norm_type 2 \
  --eval-freq 1000 --eval-time-limit 3600 --core-steps 4 \
  --expert-exploration-prob 0.0 --priority_alpha 0.5 --priority_beta 0.5 \
  --e2v-aggregator sum --n_hidden 1 --hidden_size 64 \
  --decoder_v_out_size 32 --decoder_e_out_size 1 --decoder_g_out_size 1 \
  --encoder_v_out_size 32 --encoder_e_out_size 32 --encoder_g_out_size 32 \
  --core_v_out_size 64 --core_e_out_size 64 --core_g_out_size 32

In [ ]:
# monitor training
%load_ext tensorboard
%tensorboard --logdir /content/neuroSAT/GQSAT/runs_gpu

## Part B — AlphaZeroSAT (GPU)
### B.1 Build the MCTSminisat env (GSL-free)

In [ ]:
import os
%cd /content/neuroSAT/AlphaZeroSAT
!PYTHON=python3 bash MCTSminisat/build_so.sh 2>&1 | tail -5
print('built:', os.path.exists('MCTSminisat/minisat/gym/_GymSolver.so'))

### B.2 Self-play + train (GPU)
MCTS self-play runs in the C++ env (CPU); the network trains on the GPU
(`--device auto`). Scale `--cycles` / `--n_batch` / `--n_repeat` for a real run.

In [ ]:
%cd /content/neuroSAT/AlphaZeroSAT
!python3 train_torch.py \
  --train_path data/uf20-91_train_v0 \
  --device auto \
  --n_batch 16 --n_repeat 10 --resign 400 \
  --cycles 10 --sl_num_steps 1000 --sl_n_batch 32 --lr 1e-3 \
  --save_path runs_gpu/az_model.pt

## 2. Persist checkpoints to Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
!mkdir -p '/content/gdrive/MyDrive/neuroSAT_ckpts'
!cp -r /content/neuroSAT/GQSAT/runs_gpu '/content/gdrive/MyDrive/neuroSAT_ckpts/gqsat_runs' 2>/dev/null; true
!cp -r /content/neuroSAT/AlphaZeroSAT/runs_gpu '/content/gdrive/MyDrive/neuroSAT_ckpts/alphazero_runs' 2>/dev/null; true
print('checkpoints copied to Drive/neuroSAT_ckpts')